# ViT-S/16: baseline 95/5

655500 train, 34500 val, по 1900/100 рисунков каждого класса. Фиксированная разбивка с seed 45.

Старт с внешних pretrained-весов; cross-entropy с label smoothing 0.05, без дополнительных loss. AdamW, batch 64, EMA 0.99. Старые модели сравниваются только на этих же 34500 val.


In [1]:
import json
from pathlib import Path

import pandas as pd
from IPython.display import display

output = Path('output/pretrained-vit-95-20261015')
config = json.loads((output / 'config.json').read_text())
config

{'model': 'vit_small_patch16_224.augreg_in21k_ft_in1k',
 'pretrained': True,
 'pretrained_config': {'url': 'https://storage.googleapis.com/vit_models/augreg/S_16-i21k-300ep-lr_0.001-aug_light1-wd_0.03-do_0.0-sd_0.0--imagenet2012-steps_20k-lr_0.03-res_224.npz',
  'hf_hub_id': 'timm/vit_small_patch16_224.augreg_in21k_ft_in1k',
  'architecture': 'vit_small_patch16_224',
  'tag': 'augreg_in21k_ft_in1k',
  'custom_load': True,
  'input_size': [3, 224, 224],
  'fixed_input_size': True,
  'interpolation': 'bicubic',
  'crop_pct': 0.9,
  'crop_mode': 'center',
  'mean': [0.5, 0.5, 0.5],
  'std': [0.5, 0.5, 0.5],
  'num_classes': 1000,
  'pool_size': None,
  'first_conv': 'patch_embed.proj',
  'classifier': 'head',
  'license': 'apache-2.0'},
 'input': 224,
 'patch': 16,
 'channels': 1,
 'inverted': True,
 'resize': 'bicubic, align_corners=False, clamp [-1,1]',
 'normalization': {'mean': [0.5, 0.5, 0.5], 'std': [0.5, 0.5, 0.5]},
 'parameters': 21601881,
 'batch_size': 64,
 'optimizer': 'AdamW',

In [2]:
scores = pd.read_csv(output / 'scores.csv')
display(scores)

,model,correct,accuracy
0,vit,26312,0.762667
1,vit_flip,26352,0.763826
2,resnet,25106,0.727710
3,resnet_flip,25409,0.736493


In [3]:
reference = json.loads((output / "reference.json").read_text())
display(pd.DataFrame(reference["references"]).T)


,correct,accuracy,source
vit_plain,26265,0.761304,output/pretrained-tta-20261015-0926/snapshot/v...
vit_flip,26363,0.764145,output/pretrained-tta-20261015-0926/snapshot/v...
efficientnet_flip,26307,0.762522,output/efficientnet-b3-resume-20261015/val_fli...


In [4]:
history = pd.read_csv(output / 'history.csv')
display(history)

,epoch,loss,train_accuracy,accuracy,ema_accuracy,examples,lr,seconds
0,1,2.126821,0.595577,0.692435,0.721275,655500,0.000099,616.862205
1,2,1.570123,0.710037,0.717130,0.740232,655500,0.000094,614.093130
2,3,1.438260,0.740568,0.735478,0.749652,655500,0.000084,613.632755
3,4,1.336769,0.765262,0.745594,0.756638,655500,0.000070,613.469812
4,5,1.239206,0.789538,0.751101,0.760377,655500,0.000055,614.022947
5,6,1.136512,0.815962,0.757536,0.762667,655500,0.000038,613.372463
6,7,1.032590,0.845265,0.759884,0.762058,655500,0.000023,614.143337
7,8,0.941632,0.873436,0.760551,0.760551,655500,0.000011,614.252428
8,9,0.875797,0.894805,0.757072,0.757855,655500,0.000004,613.964962
9,10,0.841203,0.907309,0.757101,0.757188,655500,0.000001,613.771958


In [5]:
summary = json.loads((output / 'summary.json').read_text())
summary

{'selected': {'model': 'vit_flip',
  'correct': 26352,
  'accuracy': 0.7638260869565218},
 'flip': True,
 'val_examples': 34500,
 'submission_rows': 172500,
 'val_above_6_threshold': True,
 'val_above_8_threshold': False}

In [6]:
verification = json.loads((output / 'verification.json').read_text())
assert verification['val_examples'] == 34500
assert verification['submission_rows'] == 172500
assert verification['submission_differences'] == 0
assert verification['accuracy'] == summary['selected']['accuracy']
verification

{'correct': 26352,
 'val_examples': 34500,
 'accuracy': 0.7638260869565218,
 'submission_rows': 172500,
 'submission_differences': 0}

In [7]:
submission = pd.read_csv(output / 'submission.csv')
assert len(submission) == 172500
assert submission['id'].tolist() == list(range(172500))
assert submission['label'].between(0, 344).all()
display(submission.head())

,id,label
0,0,278
1,1,136
2,2,292
3,3,187
4,4,64
